# Análise de Sentimentos em Dados de Saúde Mental com Técnicas de Text Mining

**Relatório técnico em formato notebook**  
Mestrado em Business Intelligence and Analytics — Text Mining para Business Analytics

Este notebook apresenta o desenvolvimento completo de um projeto de Text Mining aplicado à **análise de sentimentos**. A estrutura foi organizada para funcionar simultaneamente como implementação em Python e como relatório técnico, cobrindo: caracterização dos dados, pré-processamento, extração/seleção de atributos, classificação supervisionada, comparação de modelos, aplicação a novos exemplos e exploração de uma abordagem baseada em LLM.


## Resumo

O objetivo deste trabalho é desenvolver um sistema de classificação automática de sentimentos a partir de textos associados ao domínio da saúde mental. O problema é formulado como uma tarefa de classificação supervisionada, em que cada comentário é associado a uma polaridade de sentimento. O processo metodológico inclui uma fase de análise exploratória, limpeza e normalização textual, vetorização com TF-IDF, seleção de atributos, balanceamento das classes e comparação de diferentes classificadores.

Para tornar o trabalho menos dependente de uma única abordagem, são ainda introduzidos elementos adicionais de originalidade: comparação estruturada entre modelos, análise de erros, experiência com diferentes representações textuais e uma secção dedicada a modelos de linguagem pré-treinados com fine-tuning.


## 1. Introdução

A análise de sentimentos é uma área de Text Mining cujo objetivo é identificar automaticamente opiniões, emoções ou polaridades expressas em texto. Em contextos digitais, esta técnica é particularmente útil porque permite transformar grandes volumes de conteúdo não estruturado em informação útil para apoio à decisão.

Neste caso de estudo, o foco está em textos relacionados com saúde mental. Este domínio é sensível, pois a linguagem utilizada pelos utilizadores pode conter sinais emocionais relevantes, ambiguidades, abreviações, emojis, ruído textual e diferenças de estilo. Por essa razão, a qualidade do pré-processamento e a escolha adequada dos modelos de classificação assumem um papel central.

A metodologia seguida neste notebook está organizada em cinco fases principais:

1. caracterização e análise exploratória dos dados;
2. pré-processamento linguístico e normalização textual;
3. criação de representações numéricas através de TF-IDF;
4. treino, avaliação e comparação de classificadores supervisionados;
5. extensão com LLM/fine-tuning e discussão dos resultados.


## 2. Trabalho Relacionado

A literatura sobre análise de sentimentos evoluiu de métodos baseados em léxicos e modelos clássicos de Machine Learning para abordagens baseadas em redes neuronais e modelos de linguagem pré-treinados. Trabalhos clássicos como Pang, Lee e Vaithyanathan (2002) demonstraram a aplicação de classificadores supervisionados à deteção de polaridade em texto. Posteriormente, Liu (2012) sistematizou a análise de sentimentos como uma área fundamental da mineração de opiniões.

Mais recentemente, modelos baseados em Transformers, como BERT, passaram a apresentar desempenhos superiores em várias tarefas de NLP, devido à capacidade de aprender representações contextuais das palavras. No entanto, modelos clássicos como Logistic Regression, SVM e Naive Bayes continuam a ser relevantes em contextos académicos e empresariais, sobretudo quando existe necessidade de interpretabilidade, menor custo computacional e facilidade de implementação.

O presente trabalho diferencia-se por combinar uma abordagem clássica, baseada em TF-IDF e classificadores supervisionados, com uma secção adicional de LLM/fine-tuning. Esta combinação permite comparar soluções tradicionais com abordagens mais recentes, mantendo uma estrutura adequada ao contexto da unidade curricular.


## 3. Caso de Estudo e Caracterização Inicial dos Dados

Nesta secção é feita a leitura do dataset e uma primeira caracterização das variáveis disponíveis. A análise inicial permite compreender a estrutura dos dados, identificar valores em falta, observar a distribuição dos sentimentos e avaliar algumas variáveis contextuais como período do dia, idade e país.


In [ ]:
# Importa o módulo 'warnings'
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Importar bibliotecas base
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re

# Bibliotecas NLP
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

# Fazer download das stopwords
nltk.download('stopwords')

# Definir a paleta de cores consistente para os sentimentos em todo o projeto
sentiment_colors = {'positive': '#2ECC71', 'neutral': '#BDC3C7', 'negative': '#E74C3C'}

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/2º Semestre/Text Mining/MentalHealth.csv", encoding='latin1')

df.head(10)

In [ ]:
# Mostrar informações gerais sobre o DataFrame
df.info()

In [ ]:
# Mostrar o número de linhas e colunas
df.shape

## 2. Pré-Processamento e Limpeza


In [ ]:
# Contar quantos valores nulos existem em cada coluna
df.isnull().sum()

In [ ]:
# O dataset tem 1 nulo na coluna 'text' e 'selected_text'
df = df.dropna(subset=['text', 'selected_text'])
df.shape

In [ ]:
# 1. Criar colunas com o tamanho do texto
df['char_count'] = df['text'].apply(lambda x: len(str(x)))
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# 2. Função para limpar o texto
stop_words = set(stopwords.words('english')) # Assume que o dataset é em inglês

def clean_text(text):
    text = str(text).lower() # Converter para minúsculas
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remover URLs
    text = re.sub(r'\@\w+|\#','', text) # Remover menções (@) e hashtags (#)
    text = re.sub(r'[^\w\s]', '', text) # Remover pontuação
    # Remover stopwords
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

# Criar uma nova coluna com o texto limpo
df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text', 'word_count']].head()

## 3. Análise Exploratória: Word Cloud


In [ ]:

all_items_clean = ' '.join(df['clean_text'])

wordcloud = WordCloud(
    width=3000,
    height=2000,
    background_color='white',
    colormap='viridis',
    collocations=False,
    max_words=200
).generate(all_items_clean)

# Mostrar a Word Cloud
plt.figure(figsize=(15, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Palavras Mais Frequentes (Texto Limpo)', fontsize=16)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
sns.histplot(data=df, x='word_count', hue='sentiment', kde=True, palette=sentiment_colors, ax=axes[0])
axes[0].set_title('Distribuição do Número de Palavras por Sentimento')
axes[0].set_xlabel('Número de Palavras')
axes[0].set_ylabel('Frequência')

# Boxplot
sns.boxplot(x='sentiment', y='word_count', data=df, palette=sentiment_colors, ax=axes[1])
axes[1].set_title('Dispersão do Número de Palavras por Sentimento')
axes[1].set_xlabel('Sentimento')
axes[1].set_ylabel('Número de Palavras')

plt.tight_layout()
plt.show()

In [ ]:
def get_top_n_words(corpus, n=None, ngram_range=(1,1)):
    vec = CountVectorizer(ngram_range=ngram_range).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

# Extrair Top 20 palavras únicas
top_unigrams = get_top_n_words(df['clean_text'], 20, (1,1))
top_bigrams = get_top_n_words(df['clean_text'], 20, (2,2))

# Converter para DataFrame para facilitar o plot
df_unigrams = pd.DataFrame(top_unigrams, columns=['Palavra', 'Contagem'])
df_bigrams = pd.DataFrame(top_bigrams, columns=['Bigrama', 'Contagem'])

# Criar os gráficos
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.barplot(x='Contagem', y='Palavra', data=df_unigrams, palette='Blues_r', ax=axes[0])
axes[0].set_title('Top 20 Palavras Mais Frequentes')

sns.barplot(x='Contagem', y='Bigrama', data=df_bigrams, palette='Oranges_r', ax=axes[1])
axes[1].set_title('Top 20 Bigramas (Pares de Palavras) Mais Frequentes')

plt.tight_layout()
plt.show()

### 3.2 Análise Temporal e Demográfica


In [ ]:
# Ordenar as fases do dia
ordem_tempo = ['morning', 'noon', 'afternoon', 'evening', 'night']

plt.figure(figsize=(10, 5))
sns.countplot(x='Time of Tweet', data=df, order=ordem_tempo, color='paleturquoise')
plt.xlabel('Time of Tweet', size=12)
plt.ylabel('Number of Comments', size=12)
plt.title('Period of the day that generates the highest number of comments', color='purple', size=14)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Pie Chart de Sentimentos
sentiment_counts = df['sentiment'].value_counts()
axes[0].pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%',
            colors=[sentiment_colors.get(x, '#333333') for x in sentiment_counts.index])
axes[0].set_title('Distribution of Sentiments')

# 2. Countplot por Age of User (Ordenado)
ordem_idades = sorted(df['Age of User'].dropna().unique())
sns.countplot(x='Age of User', data=df, order=ordem_idades, color='cornflowerblue', ax=axes[1])
axes[1].set_title('Reviews by Age of User')
axes[1].tick_params(axis='x', rotation=45)

# 3. Age of User vs Sentimento
sns.countplot(x='Age of User', hue="sentiment", data=df, order=ordem_idades, palette=sentiment_colors, ax=axes[2])
axes[2].set_title('Sentiment Distribution by Age of User')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Contabilizar o número de comentários por faixa etária
print(df["Age of User"].value_counts())

### Síntese da Análise Exploratória

A análise exploratória permite retirar três conclusões importantes para a fase de modelação. Primeiro, a distribuição dos sentimentos deve ser analisada antes do treino, porque desequilíbrios entre classes podem influenciar o comportamento dos classificadores. Segundo, o comprimento dos textos e as palavras mais frequentes ajudam a perceber se existe ruído, repetição ou termos demasiado genéricos. Terceiro, variáveis contextuais como país, idade ou período do dia podem ser úteis para interpretação, embora o modelo principal deste trabalho se concentre no conteúdo textual.

Esta secção responde à componente de **Análise Exploratória dos Dados** solicitada no enunciado e prepara a decisão metodológica das fases seguintes.


In [ ]:
# Sentimento por Período do Dia
plt.figure(figsize=(12, 6))
sns.countplot(x='Time of Tweet', hue='sentiment', data=df, order=ordem_tempo, palette=sentiment_colors)
plt.title('Distribuição de Sentimento ao Longo do Dia', fontsize=14)
plt.xlabel('Período do Dia')
plt.ylabel('Número de Tweets')
plt.show()

### 5. Distribuição Geográfica (Extra)


In [ ]:
# Contabilizar o número de ocorrências por País (Top 10)
country_counts = df['Country'].value_counts().head(10)
print(country_counts)

In [ ]:
# Top 10 países com mais tweets
top_countries = df['Country'].value_counts().head(10).index

# Filtrar o dataset apenas para esses 10 países
df_top_countries = df[df['Country'].isin(top_countries)]

country_sentiment = pd.crosstab(df_top_countries['Country'], df_top_countries['sentiment'], normalize='index') * 100

# Ordenar pelos países que têm maior percentagem de sentimento negativo
country_sentiment = country_sentiment.sort_values(by='negative', ascending=False)

country_sentiment.plot(kind='bar', stacked=True, figsize=(14, 7), color=[sentiment_colors.get(col) for col in country_sentiment.columns])
plt.title('Distribuição de Sentimentos nos Top 10 Países (100% Stacked)', fontsize=15)
plt.xlabel('País', fontsize=12)
plt.ylabel('Percentagem (%)', fontsize=12)
plt.legend(title='Sentimento', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 6. Pré-Processamento Avançado para Text Mining

Nesta secção é implementada a fase de pré-processamento textual do projeto.
As técnicas aplicadas incluem:

- Remoção da classe neutra
- Identificação de NER
- Remoção de HTML tags e URLs
- Expansão de chatwords
- Expansão de emoticons
- Conversão de emojis em texto
- Conversão para minúsculas
- Expansão de contrações
- Remoção de pontuação
- Remoção de stopwords
- Filtragem de palavras válidas
- Lematização

No final, são ainda aplicadas técnicas de balanceamento de classes (**oversampling** e **undersampling**) e seleção dos melhores atributos (**feature selection**).


In [ ]:
!pip -q install contractions emoji beautifulsoup4 imbalanced-learn spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
# IMPORTS E CONFIGURAÇÃO
import re
import html
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nltk
import emoji
import contractions
import spacy

from bs4 import BeautifulSoup
from html import unescape
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, words
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.utils import resample
from imblearn.over_sampling import RandomOverSampler

# Downloads NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('words')
nltk.download('wordnet')
nltk.download('omw-1.4')

# spaCy
nlp = spacy.load("en_core_web_sm")

# Configuração principal
TEXT_COL = "text"
TARGET_COL = "sentiment"

# Cópia de trabalho
review_df = df.copy()

review_df[TEXT_COL] = review_df[TEXT_COL].astype(str)

print("Shape inicial:", review_df.shape)
print("Colunas disponíveis:", list(review_df.columns))
display(review_df[[TEXT_COL, TARGET_COL]].head(5))

## 6.1 Remoção da Classe Neutra

Para simplificar o problema de classificação, são removidas as reviews com sentimento **neutral**.  
Desta forma, o modelo passa a trabalhar apenas com duas classes:

- **positive**
- **negative**


In [ ]:
# Remover reviews com sentimento neutro
review_df = review_df[review_df[TARGET_COL] != 'neutral'].copy()

print("Distribuição após remoção da classe neutral:")
print(review_df[TARGET_COL].value_counts())
display(review_df[[TEXT_COL, TARGET_COL]].head(10))

## 6.2 Identificação de NER (Named Entity Recognition)

Nesta etapa são identificadas entidades nomeadas no texto, tais como pessoas, organizações, localizações e datas.


In [ ]:
def extract_named_entities(texts, model, batch_size=32):
    entities_list = []
    for doc in model.pipe(texts, batch_size=batch_size):
        entities_list.append([(ent.text, ent.label_) for ent in doc.ents])
    return entities_list

# Aplicar NER
review_df['named_entities'] = extract_named_entities(review_df[TEXT_COL].fillna(""), nlp)

display(review_df[[TEXT_COL, 'named_entities']].head(10))

## 6.3 Remoção de elementos HTML

Antes da normalização textual, procede-se à remoção de etiquetas HTML e entidades codificadas, uma vez que estes elementos introduzem ruído e não acrescentam valor semântico ao processo de classificação.


In [ ]:
review_df['text_without_html'] = review_df[TEXT_COL].apply(unescape)
review_df['text_without_html'] = review_df['text_without_html'].apply(
    lambda x: BeautifulSoup(str(x), "html.parser").get_text(separator=" ")
)

display(review_df[[TEXT_COL, 'text_without_html']].head(10))

## 6.4 Remoção de URLs

Nesta etapa são removidos links, endereços `www` e referências URL, por se tratarem de elementos com pouco valor semântico para a análise de sentimentos.


In [ ]:
review_df['text_without_urls'] = review_df['text_without_html'].str.replace(
    r'http\S+|www\S+|https\S+',
    '',
    regex=True
)

display(review_df[['text_without_html', 'text_without_urls']].head(10))

## 6.5 Expansão de Chatwords

Nesta etapa são identificadas abreviações típicas da comunicação digital e substituídas pela sua forma completa.  
Exemplos: `FYI` → *For Your Information*, `BRB` → *Be Right Back*, `OMG` → *Oh My God*.


In [ ]:
chatwords = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "LOL": "laugh out loud",
    "BTW": "By The Way",
    "BRB": "Be Right Back",
    "OMG": "Oh My God",
    "IDK": "I do not know",
    "FYI": "For Your Information",
    "TYT": "Take Your Time",
    "THX": "Thanks",
    "SPK": "Speak"
}

def expand_chatwords(text):
    text = str(text)
    for chatword, meaning in chatwords.items():
        text = re.sub(rf'\b{re.escape(chatword)}\b', meaning, text)
        text = re.sub(rf'\b{re.escape(chatword.lower())}\b', meaning.lower(), text)
    return text

review_df['text_without_chatwords'] = review_df['text_without_urls'].apply(expand_chatwords)

display(review_df[['text_without_urls', 'text_without_chatwords']].head(10))

## 6.6 Expansão de Emoticons

Os emoticons presentes nas reviews são convertidos em descrições textuais equivalentes, preservando o valor emocional da mensagem.


In [ ]:
emoticons_dict = {
    ":-)": " happy face ",
    ":)": " happy face ",
    ":-(": " sad face ",
    ":(": " sad face ",
    ":-D": " laughing face ",
    ":D": " laughing face ",
    ";-)": " winking face ",
    ";)": " winking face ",
    ":-/": " confused face ",
    ":/": " confused face "
}

def expand_emoticons(text):
    text = str(text)
    for emoticon, meaning in emoticons_dict.items():
        text = text.replace(emoticon, meaning)
    return text

review_df['text_without_emoticons'] = review_df['text_without_chatwords'].apply(expand_emoticons)

display(review_df[['text_without_chatwords', 'text_without_emoticons']].head(10))

## 6.7 Conversão de Emojis em Texto

Os emojis são convertidos em descrições textuais, permitindo preservar informação semântica e afetiva relevante para a análise de sentimentos.


In [ ]:
review_df['text_no_emojis'] = review_df['text_without_emoticons'].apply(
    lambda x: emoji.demojize(str(x), delimiters=(" ", " "))
)

display(review_df[['text_without_emoticons', 'text_no_emojis']].head(10))

## 6.8 Conversão para Minúsculas

A conversão para minúsculas uniformiza a representação textual e evita que palavras semanticamente idênticas sejam tratadas como diferentes apenas por causa da capitalização.


In [ ]:
review_df['lower_text'] = review_df['text_no_emojis'].str.lower()

display(review_df[['text_no_emojis', 'lower_text']].head(10))

## 6.9 Expansão de Contrações

As contrações em inglês são expandidas para a sua forma completa, por exemplo: `isn't` → `is not`, `don't` → `do not`.


In [ ]:
review_df['text_without_contractions'] = review_df['lower_text'].apply(contractions.fix)

display(review_df[['lower_text', 'text_without_contractions']].head(10))

## 6.10 Remoção de Pontuação

São removidos sinais de pontuação e caracteres especiais, preservando apenas letras, números e espaços.


In [ ]:
review_df['text_without_punctuation'] = review_df['text_without_contractions'].str.replace(
    r'[^\w\s]',
    ' ',
    regex=True
).str.replace(r'\s+', ' ', regex=True).str.strip()

display(review_df[['text_without_contractions', 'text_without_punctuation']].head(10))

## 6.11 Remoção de StopWords

As *stopwords* são palavras muito frequentes, mas com baixo poder discriminativo para tarefas de classificação de sentimentos.  
Nesta etapa são removidas da representação textual.


In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = word_tokenize(str(text))
    filtered_words = [word for word in tokens if word.lower() not in stop_words]
    return ' '.join(filtered_words)

review_df['text_without_stopwords'] = review_df['text_without_punctuation'].apply(remove_stopwords)

display(review_df[['text_without_punctuation', 'text_without_stopwords']].head(10))

## 6.12 Filtragem de Palavras Válidas

Nesta fase são mantidas apenas palavras existentes num dicionário de referência da língua inglesa, reduzindo ruído associado a erros ortográficos, símbolos residuais ou tokens pouco informativos.


In [ ]:
valid_words = set(word.lower() for word in words.words())

def keep_valid_words(text):
    tokens = word_tokenize(str(text))
    filtered_tokens = [token for token in tokens if token.lower() in valid_words]
    return ' '.join(filtered_tokens)

review_df['text_valid_words'] = review_df['text_without_stopwords'].apply(keep_valid_words)

display(review_df[['text_without_stopwords', 'text_valid_words']].head(10))

## 6.13 Lematização

A lematização reduz cada palavra à sua forma base (*lemma*), preservando melhor o significado linguístico.  
Para garantir estabilidade na execução do notebook, é utilizada a lematização simples com `WordNetLemmatizer`.


In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(str(text))
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(lemmas)

review_df['text_preprocessed'] = review_df['text_valid_words'].apply(lemmatize_text)

display(review_df[[TEXT_COL, 'text_preprocessed', TARGET_COL]].head(10))

## 6.14 Resultado Final do Pré-Processamento

A coluna `text_preprocessed` contém o texto final pronto para a fase de vetorização, balanceamento e modelação.


In [ ]:
print("Número de linhas após pré-processamento:", len(review_df))
print("Valores nulos em text_preprocessed:", review_df['text_preprocessed'].isnull().sum())
print("\nDistribuição atual das classes:")
print(review_df[TARGET_COL].value_counts())

display(review_df[[TEXT_COL, 'text_preprocessed', TARGET_COL]].head(10))

# 7. Técnicas de Balanceamento

Após o pré-processamento, avalia-se o equilíbrio entre as classes.  
São aplicadas duas estratégias:

- **Oversampling**: aumenta artificialmente a classe minoritária  
- **Undersampling**: reduz artificialmente a classe maioritária


In [ ]:
print("Distribuição original das classes:")
print(review_df[TARGET_COL].value_counts())

## 7.1 Oversampling

Nesta abordagem, a classe minoritária é aumentada até igualar o número de observações da classe maioritária.


In [ ]:
ros = RandomOverSampler(random_state=42)

X_over, y_over = ros.fit_resample(review_df[['text_preprocessed']], review_df[TARGET_COL])

df_over = pd.DataFrame({
    'text_preprocessed': X_over['text_preprocessed'],
    TARGET_COL: y_over
})

print("Distribuição após oversampling:")
print(df_over[TARGET_COL].value_counts())
display(df_over.head(10))

## 7.2 Undersampling

Nesta abordagem, a classe maioritária é reduzida até igualar o número de observações da classe minoritária.


In [ ]:
class_counts = review_df[TARGET_COL].value_counts()
majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()

df_majority = review_df[review_df[TARGET_COL] == majority_class]
df_minority = review_df[review_df[TARGET_COL] == minority_class]

df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

df_under = pd.concat([df_majority_downsampled, df_minority]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribuição após undersampling:")
print(df_under[TARGET_COL].value_counts())
display(df_under.head(10))

# 8. Feature Selection – Seleção dos Melhores Atributos

Após o pré-processamento e o balanceamento, procede-se à vetorização do texto com **TF-IDF** e à seleção dos atributos mais relevantes com o teste **Qui-Quadrado (chi2)**.


## 8.1 Vetorização com TF-IDF

Para a fase de seleção de atributos, foi escolhido o dataset balanceado por **oversampling**, uma vez que preserva toda a informação da classe maioritária.


In [ ]:
review_final = df_over.copy()

vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer.fit_transform(review_final['text_preprocessed'])
y_final = review_final[TARGET_COL]

print("Shape da matriz TF-IDF:", X_tfidf.shape)

## 8.2 Seleção dos melhores atributos com Qui-Quadrado

A técnica `SelectKBest` com `chi2` permite identificar os termos com maior relação estatística com a variável alvo.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
k_features = min(1000, X_tfidf.shape[1])

selector = SelectKBest(score_func=chi2, k=k_features)
X_selected = selector.fit_transform(X_tfidf, y_final)

selected_mask = selector.get_support()
best_features = feature_names[selected_mask]

print("Número de features selecionadas:", len(best_features))
print("\nPrimeiras 30 melhores features:")
print(best_features[:30])
print("\nShape da matriz reduzida:", X_selected.shape)

## 8.3 Resumo dos objetos finais

No final desta fase ficam disponíveis os seguintes objetos principais para a etapa de classificação:

- `review_df` → dataset pré-processado
- `df_over` → dataset balanceado com oversampling
- `df_under` → dataset balanceado com undersampling
- `review_final` → dataset escolhido para modelação
- `X_tfidf` → matriz TF-IDF
- `X_selected` → matriz reduzida após feature selection
- `best_features` → lista dos atributos mais relevantes


In [ ]:
print("Objetos principais disponíveis:")
print("- review_df")
print("- df_over")
print("- df_under")
print("- review_final")
print("- X_tfidf")
print("- X_selected")
print("- best_features")

# 9. Classificação – Criação do Modelo

Nesta fase, o texto já pré-processado é convertido para uma representação vetorial adequada à classificação supervisionada.

Seguindo a lógica, utiliza-se **TF-IDF** para representar os documentos e, de seguida, procede-se ao treino e avaliação de classificadores supervisionados para análise de sentimentos. A abordagem inclui:

- criação da matriz documento-termo;
- divisão em treino e teste;
- treino do modelo;
- avaliação com métricas de classificação;
- aplicação do modelo a um novo exemplo.


## 9.1. Importação das bibliotecas para classificação

São importadas as bibliotecas necessárias para:
- divisão treino/teste;
- treino de classificadores;
- validação cruzada;
- cálculo das métricas de avaliação;
- visualização da matriz de confusão.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    cohen_kappa_score,
    ConfusionMatrixDisplay
)


## 9.2. Criação da Matriz Documento-Termo com TF-IDF

Após a seleção dos melhores atributos com Qui-Quadrado, é criada uma nova representação TF-IDF usando apenas o vocabulário mais relevante.

Desta forma, o modelo é treinado com uma matriz documento-termo mais compacta e discriminativa.


In [ ]:
# Criar novo vectorizador apenas com as melhores features
vectorizer_model = TfidfVectorizer(vocabulary=list(best_features))

# Gerar a matriz documento-termo final para modelação
X_model = vectorizer_model.fit_transform(review_final['text_preprocessed'])

# Definir variável alvo
y_model = review_final[TARGET_COL]

print("Shape da matriz final para classificação:", X_model.shape)
print("\nDistribuição das classes no dataset final:")
print(y_model.value_counts())


## 9.3. Divisão em Dados de Treino e Teste

Tal como referido nas aulas, utiliza-se uma divisão **estratificada de 70% para treino e 30% para teste**, preservando a proporção das classes em ambos os subconjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.30,
    random_state=0,
    shuffle=True,
    stratify=y_model
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nDistribuição y_train:")
print(y_train.value_counts())
print("\nDistribuição y_test:")
print(y_test.value_counts())


## 9.4. Treino do Modelo de Classificação

Os classificadores mais usados em análise de sentimentos incluem **SVM, Logistic Regression, K-NN, Decision Trees** e outros modelos supervisionados.

Neste notebook, o classificador principal selecionado é o **SVM**, mantendo-se no código outras alternativas para testes comparativos.


In [ ]:
np.random.seed(0)

# Classificador principal
classifier = SVC(kernel='linear', probability=True, random_state=0)

# Outros classificadores possíveis para comparação
# classifier = LogisticRegression(max_iter=2000, random_state=0)
# classifier = MultinomialNB()
# classifier = KNeighborsClassifier(n_neighbors=5)
# classifier = DecisionTreeClassifier(random_state=0)
# classifier = RandomForestClassifier(random_state=0)
# classifier = AdaBoostClassifier(random_state=0)
# classifier = GradientBoostingClassifier(random_state=0)

# Treinar modelo
classifier.fit(X_train, y_train)

print("Classificador treinado com sucesso:")
print(classifier)


## 9.5. Validação Cruzada

A **K-fold Cross Validation** é uma técnica importante para avaliar o desempenho do modelo e reduzir o risco de overfitting.

Aqui é utilizada uma validação cruzada estratificada com **5 folds** sobre o conjunto total de dados.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(classifier, X_model, y_model, cv=cv, scoring='accuracy')

print("Accuracy por fold:", np.round(cv_scores, 4))
print("Accuracy média CV:", round(cv_scores.mean(), 4))
print("Desvio padrão CV:", round(cv_scores.std(), 4))


# 10. Avaliação do Modelo

Após o treino, o desempenho do classificador é avaliado no conjunto de teste com base na matriz de confusão e em métricas como **accuracy, precision, recall, F1-score, specificity** e **coeficiente Kappa**.


## 10.1. Previsões no Conjunto de Teste


In [ ]:
y_pred = classifier.predict(X_test)

print("Primeiras 20 previsões:")
print(y_pred[:20])


## 10.2. Matriz de Confusão


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=classifier.classes_)
cm


In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classifier.classes_)
disp.plot(cmap='Blues')
plt.title("Matriz de Confusão")
plt.show()


In [ ]:
# Considera-se 'negative' como classe negativa e 'positive' como classe positiva
labels_order = list(classifier.classes_)
print("Ordem das classes no modelo:", labels_order)

if labels_order == ['negative', 'positive']:
    tn, fp, fn, tp = cm.ravel()
elif labels_order == ['positive', 'negative']:
    tp, fn, fp, tn = cm.ravel()
else:
    tn, fp, fn, tp = cm.ravel()

print("True Negative (TN):", tn)
print("False Positive (FP):", fp)
print("False Negative (FN):", fn)
print("True Positive (TP):", tp)


## 10.3. Métricas de Avaliação


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='positive')
recall = recall_score(y_test, y_pred, pos_label='positive')
f1 = f1_score(y_test, y_pred, pos_label='positive')
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
kappa = cohen_kappa_score(y_test, y_pred)
error_rate = 1 - accuracy

metrics_df = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall / Sensitivity', 'Specificity', 'F1-Score', 'Kappa', 'Error Rate'],
    'Valor': [accuracy, precision, recall, specificity, f1, kappa, error_rate]
})

display(metrics_df)


In [ ]:
print("Relatório de Classificação:\n")
print(classification_report(y_test, y_pred))


## 10.4. Resumo Interpretativo

- **Accuracy** mede a proporção total de classificações corretas.
- **Precision** mede a proporção de previsões positivas que estavam corretas.
- **Recall / Sensitivity** mede a capacidade de identificar corretamente os exemplos positivos.
- **Specificity** mede a capacidade de identificar corretamente os exemplos negativos.
- **F1-Score** representa o equilíbrio entre precision e recall.
- **Kappa** mede a concordância entre previsões e valores reais corrigindo o efeito do acaso.


# 11. Comparação Estruturada entre Classificadores

Para tornar a fase de classificação mais robusta e original, nesta secção são testados vários classificadores supervisionados sobre a mesma representação TF-IDF. Em vez de assumir à partida que o SVM é o melhor, os modelos são avaliados de forma comparável e os resultados são reunidos numa tabela final.

Os modelos comparados são:
- SVM
- Logistic Regression
- Multinomial Naive Bayes
- K-Nearest Neighbors
- Decision Tree
- Random Forest

A seleção do melhor modelo será feita com base no **F1-Score**, complementado pela análise de **accuracy**, **recall**, **precision**, **specificity** e **kappa**. Esta abordagem acrescenta originalidade ao trabalho e permite justificar tecnicamente a escolha do classificador final.


## 11.1 Definição dos Classificadores Candidatos


In [ ]:
candidate_models = {
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=0),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=0),
    'Multinomial NB': MultinomialNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=0),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=0)
}

print("Modelos a comparar:")
for model_name in candidate_models.keys():
    print("-", model_name)

## 11.2 Treino Individual de Cada Classificador

Cada classificador é treinado separadamente sobre os mesmos dados de treino e testado sobre o mesmo conjunto de teste. Além das métricas no conjunto de teste, é também calculada a accuracy média por validação cruzada estratificada com 5 folds.


In [ ]:
comparison_results = []
trained_models = {}

cv_compare = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    y_pred_model = model.predict(X_test)

    cm_model = confusion_matrix(y_test, y_pred_model, labels=['negative', 'positive'])
    tn_m, fp_m, fn_m, tp_m = cm_model.ravel()

    acc_m = accuracy_score(y_test, y_pred_model)
    prec_m = precision_score(y_test, y_pred_model, pos_label='positive', zero_division=0)
    rec_m = recall_score(y_test, y_pred_model, pos_label='positive', zero_division=0)
    f1_m = f1_score(y_test, y_pred_model, pos_label='positive', zero_division=0)
    spec_m = tn_m / (tn_m + fp_m) if (tn_m + fp_m) > 0 else 0
    kappa_m = cohen_kappa_score(y_test, y_pred_model)

    cv_scores_model = cross_val_score(model, X_model, y_model, cv=cv_compare, scoring='accuracy')

    comparison_results.append({
        'Modelo': model_name,
        'Accuracy_Test': acc_m,
        'Precision_Test': prec_m,
        'Recall_Test': rec_m,
        'Specificity_Test': spec_m,
        'F1_Test': f1_m,
        'Kappa_Test': kappa_m,
        'CV_Accuracy_Mean': cv_scores_model.mean(),
        'CV_Accuracy_STD': cv_scores_model.std()
    })

    trained_models[model_name] = model

    print(f"\n{'='*70}")
    print(f"MODELO: {model_name}")
    print(f"{'='*70}")
    print("Accuracy (teste):", round(acc_m, 4))
    print("Precision:", round(prec_m, 4))
    print("Recall:", round(rec_m, 4))
    print("Specificity:", round(spec_m, 4))
    print("F1-Score:", round(f1_m, 4))
    print("Kappa:", round(kappa_m, 4))
    print("CV Accuracy média:", round(cv_scores_model.mean(), 4))
    print("CV Accuracy std:", round(cv_scores_model.std(), 4))

## 11.3 Tabela Comparativa Consolidada


In [ ]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by=['F1_Test', 'Kappa_Test', 'CV_Accuracy_Mean'],
    ascending=False
).reset_index(drop=True)

display(comparison_df.style.format({
    'Accuracy_Test': '{:.4f}',
    'Precision_Test': '{:.4f}',
    'Recall_Test': '{:.4f}',
    'Specificity_Test': '{:.4f}',
    'F1_Test': '{:.4f}',
    'Kappa_Test': '{:.4f}',
    'CV_Accuracy_Mean': '{:.4f}',
    'CV_Accuracy_STD': '{:.4f}'
}))

## 11.4 Identificação do Melhor Modelo


In [ ]:
best_model_name = comparison_df.loc[0, 'Modelo']
best_model = trained_models[best_model_name]

print("Melhor modelo selecionado:", best_model_name)
print(best_model)

## 11.5 Visualização Comparativa dos Resultados

Para facilitar a interpretação, apresenta-se um gráfico comparativo do desempenho dos classificadores em termos de F1-Score no conjunto de teste.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparison_df['Modelo'], comparison_df['F1_Test'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('F1-Score (Teste)')
plt.title('Comparação dos Classificadores por F1-Score')
plt.tight_layout()
plt.show()

## 11.6 Observação sobre a Escolha Final

A escolha do melhor modelo não deve basear-se apenas na accuracy. Em problemas de classificação binária, especialmente em análise de sentimentos, o **F1-Score** e o **Kappa** ajudam a avaliar melhor o equilíbrio entre acertos e erros do classificador.

Por essa razão, o modelo final adotado para aplicação a novos exemplos será o classificador que apresentar melhor desempenho global na tabela comparativa.


## 11.7 Análise de Erros do Melhor Modelo

Para além das métricas globais, é importante observar exemplos concretos em que o modelo falha. Esta análise ajuda a perceber se os erros estão associados a textos ambíguos, curtos, irónicos, com pouca informação emocional ou com ruído linguístico.


In [ ]:
# Identificar exemplos mal classificados pelo melhor modelo
error_analysis_df = pd.DataFrame({
    'texto_preprocessado': review_final.iloc[y_test.index if hasattr(y_test, 'index') else range(len(y_test))]['text_preprocessed'].values if hasattr(y_test, 'index') else [''] * len(y_test),
    'classe_real': list(y_test),
    'classe_prevista': list(best_model.predict(X_test))
})

errors_df = error_analysis_df[error_analysis_df['classe_real'] != error_analysis_df['classe_prevista']]
print('Número de erros encontrados:', len(errors_df))
display(errors_df.head(10))


## 11.8 Experiência de Originalidade: TF-IDF com N-grams

Como elemento adicional de inovação, testa-se uma representação alternativa com **unigramas e bigramas**. A utilização de bigramas permite capturar pequenas expressões compostas, como *not good*, *very happy* ou *feel bad*, que podem ser relevantes para análise de sentimentos e que se perdem quando se consideram apenas palavras isoladas.


In [ ]:
from sklearn.pipeline import Pipeline

ngram_experiments = {
    'TF-IDF unigramas + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 1))),
        ('clf', LogisticRegression(max_iter=2000, random_state=0))
    ]),
    'TF-IDF unigramas+bigramas + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=8000, ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=2000, random_state=0))
    ])
}

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    review_final['text_preprocessed'],
    review_final[TARGET_COL],
    test_size=0.30,
    random_state=0,
    stratify=review_final[TARGET_COL]
)

ngram_results = []
for exp_name, pipe in ngram_experiments.items():
    pipe.fit(X_text_train, y_text_train)
    pred = pipe.predict(X_text_test)
    ngram_results.append({
        'Experiência': exp_name,
        'Accuracy': accuracy_score(y_text_test, pred),
        'Precision': precision_score(y_text_test, pred, pos_label='positive', zero_division=0),
        'Recall': recall_score(y_text_test, pred, pos_label='positive', zero_division=0),
        'F1-Score': f1_score(y_text_test, pred, pos_label='positive', zero_division=0),
        'Kappa': cohen_kappa_score(y_text_test, pred)
    })

ngram_results_df = pd.DataFrame(ngram_results).sort_values(by='F1-Score', ascending=False)
display(ngram_results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'Kappa': '{:.4f}'
}))


### Interpretação da experiência com N-grams

Se a versão com bigramas apresentar melhor desempenho, isso sugere que algumas expressões compostas têm valor discriminativo para a classificação de sentimentos. Caso contrário, a representação mais simples com unigramas poderá ser preferível por ser menos complexa e mais fácil de interpretar.


# 12. Aplicação do Melhor Modelo a um Novo Exemplo

Depois de selecionado o melhor classificador com base nos resultados comparativos, o passo seguinte consiste em aplicar esse modelo a um novo comentário ainda não visto.


## 12.1 Funções de Pré-Processamento para Novo Comentário


In [ ]:
chat_words_map = {
    "FYI": "for your information",
    "BRB": "be right back",
    "THX": "thanks",
    "AFAIK": "as far as i know",
    "ASAP": "as soon as possible",
    "LOL": "laugh out loud",
    "BTW": "by the way",
    "OMG": "oh my god",
    "IDK": "i do not know"
}

emoticon_map = {
    ":-)": "happy face",
    ":)": "happy face",
    ";-)": "wink face",
    ";)": "wink face",
    ":-(": "sad face",
    ":(": "sad face",
    ":-D": "laughing face",
    ":D": "laughing face"
}

stop_words_set = set(stopwords.words('english'))
valid_words_set = set(word.lower() for word in words.words())
lemmatizer_new = WordNetLemmatizer()

def remove_html_and_urls(text):
    text = BeautifulSoup(str(text), "html.parser").get_text()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    return unescape(text)

def expand_chatwords(text):
    tokens = str(text).split()
    expanded = [chat_words_map.get(tok.upper(), tok) for tok in tokens]
    return ' '.join(expanded)

def expand_emoticons(text):
    for emo, meaning in emoticon_map.items():
        text = text.replace(emo, f" {meaning} ")
    return text

def convert_emojis_to_text(text):
    text = emoji.demojize(str(text), delimiters=(" ", " "))
    text = text.replace("_", " ").replace(":", " ")
    return text

def to_lowercase(text):
    return str(text).lower()

def expand_contractions_text(text):
    return contractions.fix(str(text))

def remove_punctuation(text):
    return str(text).translate(str.maketrans('', '', string.punctuation))

def remove_stopwords_text(text):
    tokens = word_tokenize(str(text))
    filtered = [w for w in tokens if w.lower() not in stop_words_set]
    return ' '.join(filtered)

def keep_valid_words_text(text):
    tokens = word_tokenize(str(text))
    filtered = [w for w in tokens if w.lower() in valid_words_set]
    return ' '.join(filtered)

def lemmatize_text_new(text):
    tokens = word_tokenize(str(text))
    lemmas = [lemmatizer_new.lemmatize(w) for w in tokens]
    return ' '.join(lemmas)

def preprocess_new_text(text):
    text = remove_html_and_urls(text)
    text = expand_chatwords(text)
    text = expand_emoticons(text)
    text = convert_emojis_to_text(text)
    text = to_lowercase(text)
    text = expand_contractions_text(text)
    text = remove_punctuation(text)
    text = remove_stopwords_text(text)
    text = keep_valid_words_text(text)
    text = lemmatize_text_new(text)
    return text

## 12.2 Classificação de um Novo Comentário


In [ ]:
novo_comentario = """OMG!!! I wasn't feeling good at first, but the support team replied ASAP :-)
Check this: https://example.com <b>Really helpful service</b> ✈️"""

print("Melhor modelo a ser usado:", best_model_name)
print("\nComentário original:")
print(novo_comentario)

novo_comentario_pp = preprocess_new_text(novo_comentario)

print("\nComentário pré-processado:")
print(novo_comentario_pp)

In [ ]:
matriz_novo_comentario = vectorizer_model.transform([novo_comentario_pp])
previsao = best_model.predict(matriz_novo_comentario)[0]

print("Previsão do sentimento:", previsao)

## 12.3 Probabilidades Associadas à Previsão

Quando o melhor modelo selecionado suporta probabilidades, estas podem ser apresentadas para complementar a interpretação da previsão. Caso o classificador não disponha de `predict_proba`, esta etapa é automaticamente adaptada.


In [ ]:
if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(matriz_novo_comentario)[0]
    prob_df = pd.DataFrame({
        'Classe': best_model.classes_,
        'Probabilidade': probs
    }).sort_values(by='Probabilidade', ascending=False).reset_index(drop=True)
    display(prob_df)
else:
    print(f"O modelo {best_model_name} não disponibiliza probabilidades com predict_proba().")

# 13. Uso de LLM com Fine-Tuning

O enunciado solicita a utilização de LLMs com fine-tuning. Para responder a esse requisito, esta secção apresenta uma abordagem com **DistilBERT**, um modelo Transformer pré-treinado, posteriormente ajustado ao dataset do projeto.

Esta experiência deve ser entendida como uma extensão ao pipeline clássico. Enquanto os modelos anteriores dependem de TF-IDF, o modelo baseado em LLM aprende representações contextuais do texto, considerando a posição e o significado das palavras no contexto da frase.

> Nota: esta secção é mais exigente computacionalmente. Recomenda-se executar no Google Colab com GPU ativa: `Runtime > Change runtime type > GPU`.


In [ ]:
# Instalação das bibliotecas necessárias para a experiência com LLM
!pip -q install transformers datasets accelerate evaluate


In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder

print('GPU disponível:', torch.cuda.is_available())


## 13.1 Preparação dos Dados para o Modelo Transformer

Para reduzir o custo computacional, é criada uma amostra estratificada do dataset. Esta decisão mantém a proporção das classes e permite demonstrar o fine-tuning de forma prática no Colab.


In [ ]:
llm_df = review_df.copy()
sample_size_per_class = min(800, llm_df[TARGET_COL].value_counts().min())

llm_sample = (
    llm_df
    .groupby(TARGET_COL, group_keys=False)
    .apply(lambda x: x.sample(sample_size_per_class, random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

label_encoder = LabelEncoder()
llm_sample['label'] = label_encoder.fit_transform(llm_sample[TARGET_COL])

train_llm, test_llm = train_test_split(
    llm_sample[[TEXT_COL, 'label']],
    test_size=0.20,
    random_state=42,
    stratify=llm_sample['label']
)

print('Distribuição treino:')
print(train_llm['label'].value_counts())

print('\nDistribuição teste:')
print(test_llm['label'].value_counts())

print('\nMapeamento das classes:')
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


## 13.2 Tokenização e Fine-Tuning do DistilBERT

O texto é convertido para o formato esperado pelo modelo Transformer através do tokenizer. De seguida, o modelo é ajustado ao problema de classificação binária do projeto.


In [ ]:
model_checkpoint = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_llm.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_llm.reset_index(drop=True))

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns([TEXT_COL])
test_dataset = test_dataset.remove_columns([TEXT_COL])
train_dataset.set_format('torch')
test_dataset.set_format('torch')

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

llm_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
def compute_metrics_llm(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'kappa': cohen_kappa_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir='./distilbert_sentiment_results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(
    model=llm_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_llm
)

trainer.train()
llm_eval_results = trainer.evaluate()
llm_eval_results


## 13.3 Comparação entre Modelo Clássico e LLM

Após o fine-tuning, os resultados do DistilBERT podem ser comparados com os modelos clássicos. Esta comparação permite discutir se o ganho de desempenho, caso exista, compensa o aumento do custo computacional e da complexidade.


In [ ]:
llm_summary = pd.DataFrame([{
    'Modelo': 'DistilBERT fine-tuned',
    'Accuracy': llm_eval_results.get('eval_accuracy'),
    'Precision': llm_eval_results.get('eval_precision'),
    'Recall': llm_eval_results.get('eval_recall'),
    'F1-Score': llm_eval_results.get('eval_f1'),
    'Kappa': llm_eval_results.get('eval_kappa')
}])

classic_best_summary = pd.DataFrame([{
    'Modelo': best_model_name,
    'Accuracy': comparison_df.loc[0, 'Accuracy_Test'],
    'Precision': comparison_df.loc[0, 'Precision_Test'],
    'Recall': comparison_df.loc[0, 'Recall_Test'],
    'F1-Score': comparison_df.loc[0, 'F1_Test'],
    'Kappa': comparison_df.loc[0, 'Kappa_Test']
}])

final_model_comparison = pd.concat([classic_best_summary, llm_summary], ignore_index=True)
display(final_model_comparison.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'Kappa': '{:.4f}'
}))


Os resultados obtidos demonstram que o modelo baseado em DistilBERT apresenta um desempenho superior relativamente ao modelo tradicional Multinomial Naive Bayes.

Em particular, o DistilBERT alcançou um F1-score de 0.8874, superando o valor de 0.8512 obtido pelo modelo Naive Bayes. Esta melhoria evidencia a capacidade dos modelos de linguagem pré-treinados em capturar relações semânticas mais complexas no texto.

Adicionalmente, destaca-se o valor elevado de precisão (0.9437), indicando que o modelo apresenta uma baixa taxa de falsos positivos, sendo altamente fiável nas suas previsões.

Embora o recall apresente apenas uma melhoria ligeira, os resultados globais confirmam a superioridade do modelo baseado em deep learning.

Por fim, o coeficiente Kappa de 0.7875 indica uma forte concordância entre as previsões do modelo e os valores reais, reforçando a robustez da abordagem adotada.

## Análise Complementar dos Resultados

Para além da comparação global entre modelos, foi realizada uma análise mais detalhada ao desempenho do modelo DistilBERT fine-tuned, uma vez que este apresentou os melhores resultados globais.

Esta análise inclui a matriz de confusão, o relatório de classificação por classe e a identificação dos principais tipos de erro. O objetivo é compreender não apenas qual o modelo com melhor desempenho, mas também em que situações o modelo apresenta maior dificuldade de classificação.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions from the trained model
predictions_output = trainer.predict(test_dataset)

# Extract true labels (y_true) and predicted labels (y_pred_bert)
y_true = predictions_output.label_ids
y_pred_logits = predictions_output.predictions
y_pred_bert = np.argmax(y_pred_logits, axis=-1)

# Ensure class_names are in the order corresponding to the numerical labels (0, 1)
# label_encoder.classes_ provides this directly if it was fit correctly.
# Mapeamento das classes: {'negative': np.int64(0), 'positive': np.int64(1)}
class_names = list(label_encoder.classes_)

cm = confusion_matrix(y_true, y_pred_bert)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

plt.figure(figsize=(6, 5))
disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de Confusão - DistilBERT Fine-tuned")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Criar dataframe com previsões
error_analysis_df = test_llm.copy().reset_index(drop=True)

error_analysis_df["real_label"] = y_true
error_analysis_df["predicted_label"] = y_pred_bert

error_analysis_df["real_sentiment"] = label_encoder.inverse_transform(error_analysis_df["real_label"])
error_analysis_df["predicted_sentiment"] = label_encoder.inverse_transform(error_analysis_df["predicted_label"])

# Filtrar apenas erros
errors_df = error_analysis_df[
    error_analysis_df["real_label"] != error_analysis_df["predicted_label"]
]

print(f"Número total de erros: {len(errors_df)}")
print(f"Percentagem de erros: {len(errors_df) / len(error_analysis_df) * 100:.2f}%")

errors_df[[TEXT_COL, "real_sentiment", "predicted_sentiment"]].head(10)

In [ ]:
import matplotlib.pyplot as plt

error_summary["confusion_type"] = (
    error_summary["real_sentiment"].astype(str)
    + " → "
    + error_summary["predicted_sentiment"].astype(str)
)

plt.figure(figsize=(8, 5))
plt.barh(error_summary["confusion_type"], error_summary["n_errors"])
plt.xlabel("Número de erros")
plt.ylabel("Tipo de confusão")
plt.title("Tipos de Erro Mais Frequentes - DistilBERT")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
most_common_error = error_summary.iloc[0]

print("Interpretação automática dos erros:")
print(
    f"O erro mais frequente ocorreu quando textos da classe "
    f"'{most_common_error['real_sentiment']}' foram classificados como "
    f"'{most_common_error['predicted_sentiment']}'. "
    f"Este tipo de erro ocorreu {most_common_error['n_errors']} vezes."
)

### Conclusão da Análise dos Resultados

Os resultados obtidos evidenciam um desempenho elevado do modelo DistilBERT fine-tuned, com elevada capacidade de classificação correta das observações.

A análise dos erros demonstra que a principal dificuldade do modelo reside na distinção entre textos positivos e negativos em contextos ambíguos, verificando-se uma maior incidência de erros do tipo positive → negative.

Apesar destas limitações, o modelo apresenta uma elevada robustez e supera claramente os modelos tradicionais, confirmando a sua adequação para tarefas de análise de sentimentos.

# 14. Conclusões

Este trabalho desenvolveu um pipeline completo de Text Mining para análise de sentimentos. A primeira fase permitiu caracterizar o dataset e identificar aspetos relevantes da distribuição dos sentimentos e da composição textual. De seguida, foi aplicado um conjunto de técnicas de pré-processamento, incluindo remoção de ruído, expansão de abreviações, tratamento de emojis/emoticons, remoção de stopwords e lematização.

Na fase de modelação, os textos foram representados através de TF-IDF e foram testados vários classificadores supervisionados. A comparação entre modelos permitiu selecionar o melhor classificador com base em métricas como accuracy, precision, recall, F1-score, specificity e Kappa. A utilização de F1-score e Kappa é particularmente importante porque evita que a escolha do modelo dependa apenas da accuracy.

Como elementos de diferenciação, foram incluídas uma análise de erros, uma experiência com n-grams e uma abordagem com LLM/fine-tuning. Estas extensões tornam o projeto mais completo e demonstram capacidade crítica na comparação entre métodos clássicos e métodos recentes de NLP.

Em termos práticos, o modelo final pode ser aplicado a novos comentários para prever automaticamente o sentimento associado. Como trabalho futuro, seria possível explorar datasets maiores, testar modelos específicos para linguagem emocional ou saúde mental, comparar embeddings pré-treinados e aprofundar a explicabilidade dos modelos.


# 15. Referências

- Pang, B., Lee, L., & Vaithyanathan, S. (2002). *Thumbs up? Sentiment classification using machine learning techniques*. Proceedings of EMNLP.
- Liu, B. (2012). *Sentiment Analysis and Opinion Mining*. Morgan & Claypool Publishers.
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL-HLT.
- Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). *Attention Is All You Need*. NeurIPS.
- Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research.
